# 04 — Pan-Cancer Cox and Landmark Analysis

Test whether the data-driven temporal transition has prognostic significance pan-cancer
using overall survival as an independent endpoint.

Two analyses:
1. **Cox regression** with `phase_early` (early recurrence within 24 months) as
   predictor of overall survival, per cancer type and pooled across cancers.
2. **24-month landmark analysis** restricted to patients surviving past the
   transition window, to test whether the phase effect persists beyond the
   trivial early-death contribution.

The recurrence endpoint is DFI where available, PFI as fallback. KIRC has too few
DFI events for stable per-cancer Cox; we report it descriptively in the landmark.


In [ ]:
import os
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test

warnings.filterwarnings("ignore")

BASE_DIR = Path(os.environ.get("COSMOS_BASE_DIR", "./data"))
RAW_DIR = BASE_DIR / "Raw Data"
INTER_DIR = BASE_DIR / "intermediates"
DAYS_PER_MO = 365.25 / 12
LANDMARK_M = 24

CANCER_TYPES = ["BRCA", "COAD", "HNSC", "KIRC", "LIHC", "LUAD", "LUSC", "STAD", "UCEC"]

CANCER_LABELS = {
    "BRCA": "Breast invasive carcinoma",
    "COAD": "Colon adenocarcinoma",
    "HNSC": "Head and neck squamous cell",
    "KIRC": "Kidney renal clear cell",
    "LIHC": "Liver hepatocellular",
    "LUAD": "Lung adenocarcinoma",
    "LUSC": "Lung squamous cell",
    "STAD": "Stomach adenocarcinoma",
    "UCEC": "Uterine corpus endometrial",
}


## Load CDR


In [ ]:
cdr_path = RAW_DIR / "tcga_clinical" / "TCGA-CDR.csv"
if not cdr_path.exists():
    cdr_path = cdr_path.with_suffix(".xlsx")
cdr = pd.read_csv(cdr_path) if cdr_path.suffix == ".csv" else pd.read_excel(cdr_path)
print(f"CDR: {len(cdr)} patients, {cdr['type'].nunique()} cancer types")


def cdr_sub(cancer):
    sub = cdr[cdr["type"] == cancer].copy()
    if "bcr_patient_barcode" in sub.columns:
        sub = sub.set_index("bcr_patient_barcode")
    sub.index = sub.index.astype(str).str[:12]
    return sub.loc[~sub.index.duplicated()]


def get_endpoint(sub, ep):
    """Extract time/event for an endpoint with day-to-month conversion."""
    tc = f"{ep}.time"
    if tc not in sub.columns:
        return None
    out = pd.DataFrame({
        "time": pd.to_numeric(sub[tc], errors="coerce") / DAYS_PER_MO,
        "event": (pd.to_numeric(sub[ep], errors="coerce") > 0).astype(float),
    }, index=sub.index).dropna()
    return out[out["time"] > 0]


## Build phase predictor and OS outcome

The phase predictor comes from the recurrence endpoint (DFI preferred, PFI as
fallback). The Cox outcome is overall survival, an independent time axis. This
separation prevents tautological inflation that would arise from using the same
endpoint for both classification and outcome.


In [ ]:
analysis_dfs = {}
for ct in CANCER_TYPES:
    sub = cdr_sub(ct)
    rec_ep = get_endpoint(sub, "DFI")
    ep_used = "DFI"
    if rec_ep is None or len(rec_ep) < 30:
        rec_ep = get_endpoint(sub, "PFI")
        ep_used = "PFI"
    if rec_ep is None or len(rec_ep) < 30:
        print(f"  {ct}: no recurrence endpoint")
        continue

    os_ep = get_endpoint(sub, "OS")
    if os_ep is None or len(os_ep) < 30:
        print(f"  {ct}: no OS data")
        continue

    phase = pd.Series(
        np.where((rec_ep["event"] == 1) & (rec_ep["time"] <= LANDMARK_M), "early", "late"),
        index=rec_ep.index, name="phase",
    )
    df = pd.DataFrame({"phase": phase}).join(
        os_ep.rename(columns={"time": "os_time", "event": "os_event"}), how="inner",
    ).dropna()
    df["phase_bin"] = (df["phase"] == "early").astype(int)

    n_os_ev = int(df["os_event"].sum())
    if n_os_ev < 15:
        print(f"  {ct}: skipped, only {n_os_ev} OS events")
        continue

    n_early = int((df["phase"] == "early").sum())
    print(f"  {ct}: n={len(df)}, OS events={n_os_ev}, "
          f"endpoint={ep_used}, early={n_early}")
    analysis_dfs[ct] = df


## Per-cancer Cox

`phase_bin` is the binary indicator of early recurrence. Cox is fitted per cancer
type with a penalizer for numerical stability.


In [ ]:
cox_results = {}
print(f"  {'Cancer':<8} {'n':>5} {'evs':>5}  {'HR':>7}  {'95% CI':>15}  {'P':>10}  {'C':>6}")
for ct, df in analysis_dfs.items():
    try:
        cph = CoxPHFitter(penalizer=0.1)
        cph.fit(
            df[["os_time", "os_event", "phase_bin"]],
            duration_col="os_time", event_col="os_event",
        )
        hr = float(np.exp(cph.params_["phase_bin"]))
        ci_l = float(np.exp(cph.confidence_intervals_.loc["phase_bin", "95% lower-bound"]))
        ci_u = float(np.exp(cph.confidence_intervals_.loc["phase_bin", "95% upper-bound"]))
        pval = float(cph.summary["p"]["phase_bin"])
        conc = float(cph.concordance_index_)
        cox_results[ct] = {
            "HR": hr, "CI_l": ci_l, "CI_u": ci_u, "P": pval, "C": conc,
            "n": len(df), "n_ev": int(df["os_event"].sum()),
            "label": CANCER_LABELS[ct],
        }
        print(f"  {ct:<8} {len(df):>5} {int(df['os_event'].sum()):>5}  "
              f"{hr:>7.2f}  [{ci_l:.2f}-{ci_u:.2f}]  {pval:>10.4f}  {conc:>6.3f}")
    except Exception as e:
        print(f"  {ct}: Cox failed ({e})")


## Pan-cancer pooled Cox


In [ ]:
if not analysis_dfs:
    raise RuntimeError("No cancer types passed the event threshold; check CDR endpoint columns")
pan_df = pd.concat(analysis_dfs.values(), ignore_index=True)
cph_pan = CoxPHFitter(penalizer=0.1)
cph_pan.fit(
    pan_df[["os_time", "os_event", "phase_bin"]],
    duration_col="os_time", event_col="os_event",
)
hr_pan = float(np.exp(cph_pan.params_["phase_bin"]))
ci_lp = float(np.exp(cph_pan.confidence_intervals_.loc["phase_bin", "95% lower-bound"]))
ci_up = float(np.exp(cph_pan.confidence_intervals_.loc["phase_bin", "95% upper-bound"]))
p_pan = float(cph_pan.summary["p"]["phase_bin"])
c_pan = float(cph_pan.concordance_index_)

cox_results["PAN"] = {
    "HR": hr_pan, "CI_l": ci_lp, "CI_u": ci_up, "P": p_pan, "C": c_pan,
    "n": len(pan_df), "n_ev": int(pan_df["os_event"].sum()),
    "label": "Pan-cancer (pooled)",
}
print(f"  PAN:    n={len(pan_df)}, events={int(pan_df['os_event'].sum())},"
      f" HR={hr_pan:.2f} [{ci_lp:.2f}-{ci_up:.2f}], P={p_pan:.2e}, C={c_pan:.3f}")


## Sensitivity: leave-one-cancer-out

Re-fit the pooled Cox excluding each cancer type in turn to test whether the
pan-cancer signal depends on any single cohort.


In [ ]:
print("Leave-one-cancer-out pan-cancer HR:")
loo_results = {}
for excluded in analysis_dfs.keys():
    loo_df = pd.concat(
        [df for ct, df in analysis_dfs.items() if ct != excluded],
        ignore_index=True,
    )
    cph = CoxPHFitter(penalizer=0.1)
    cph.fit(loo_df[["os_time", "os_event", "phase_bin"]],
            duration_col="os_time", event_col="os_event")
    hr = float(np.exp(cph.params_["phase_bin"]))
    p = float(cph.summary["p"]["phase_bin"])
    loo_results[excluded] = {"HR": hr, "P": p, "n": len(loo_df)}
    print(f"  excluding {excluded}: HR={hr:.2f}, P={p:.2e}, n={len(loo_df)}")


## 24-month landmark analysis

Restrict to patients who survived at least 24 months and reset the time axis to
the landmark. This isolates the persistent prognostic effect of the phase definition
after the transition boundary.


In [ ]:
def build_landmark_df():
    """Build a landmark dataframe with phase predictor and OS outcome reset
    to the 24-month landmark."""
    rows = []
    for ct in CANCER_TYPES:
        sub = cdr_sub(ct)
        rec_ep = get_endpoint(sub, "DFI")
        ep_used = "DFI"
        if rec_ep is None or len(rec_ep) < 30:
            rec_ep = get_endpoint(sub, "PFI")
            ep_used = "PFI"
        if rec_ep is None:
            continue
        os_ep = get_endpoint(sub, "OS")
        if os_ep is None:
            continue
        phase = pd.Series(
            np.where(
                (rec_ep["event"] == 1) & (rec_ep["time"] <= LANDMARK_M),
                "early", "late",
            ),
            index=rec_ep.index, name="phase",
        )
        df = pd.DataFrame({"phase": phase, "cancer_type": ct}).join(
            os_ep.rename(columns={"time": "os_time", "event": "os_event"}), how="inner",
        ).dropna()
        df = df[df["os_time"] >= LANDMARK_M].copy()
        df["landmark_time"] = df["os_time"] - LANDMARK_M
        df["landmark_event"] = df["os_event"]
        df["phase_early"] = (df["phase"] == "early").astype(int)
        rows.append(df)
    return pd.concat(rows, ignore_index=True)


lm_df = build_landmark_df()
print(f"Landmark cohort: n={len(lm_df)}, post-landmark events={int(lm_df['landmark_event'].sum())}")
print(f"  Early phase: n={int(lm_df['phase_early'].sum())}")
print(f"  Late phase:  n={int((lm_df['phase_early']==0).sum())}")


In [ ]:
# Per-cancer landmark phase Cox
print("Per-cancer post-landmark phase Cox:")
lm_per_cancer = {}
for ct in lm_df["cancer_type"].unique():
    sub = lm_df[lm_df["cancer_type"] == ct].copy()
    n_e = int(sub["phase_early"].sum())
    n_l = int((sub["phase_early"] == 0).sum())
    if n_e < 3 or n_l < 5:
        print(f"  {ct}: insufficient events (early={n_e}, late={n_l})")
        continue
    try:
        cph = CoxPHFitter(penalizer=0.1)
        cph.fit(
            sub[["landmark_time", "landmark_event", "phase_early"]],
            duration_col="landmark_time", event_col="landmark_event",
        )
        hr = float(np.exp(cph.params_["phase_early"]))
        ci_l = float(np.exp(cph.confidence_intervals_.loc["phase_early", "95% lower-bound"]))
        ci_u = float(np.exp(cph.confidence_intervals_.loc["phase_early", "95% upper-bound"]))
        lr = logrank_test(
            sub.loc[sub["phase_early"] == 1, "landmark_time"],
            sub.loc[sub["phase_early"] == 0, "landmark_time"],
            event_observed_A=sub.loc[sub["phase_early"] == 1, "landmark_event"],
            event_observed_B=sub.loc[sub["phase_early"] == 0, "landmark_event"],
        )
        p = float(lr.p_value)
        lm_per_cancer[ct] = {
            "HR": hr, "CI_l": ci_l, "CI_u": ci_u, "P": p,
            "n_early": n_e, "n_late": n_l, "n_events": int(sub["landmark_event"].sum()),
        }
        print(f"  {ct}: HR={hr:.2f} [{ci_l:.2f}-{ci_u:.2f}], P={p:.4f}, "
              f"early/late={n_e}/{n_l}")
    except Exception as e:
        print(f"  {ct}: failed ({e})")


## Pan-cancer landmark Cox (stratified by cancer type)


In [ ]:
cph_lm = CoxPHFitter(penalizer=0.1)
cph_lm.fit(
    lm_df[["landmark_time", "landmark_event", "phase_early", "cancer_type"]],
    duration_col="landmark_time", event_col="landmark_event",
    strata=["cancer_type"],
)
hr_lm = float(np.exp(cph_lm.params_["phase_early"]))
ci_l_lm = float(np.exp(cph_lm.confidence_intervals_.loc["phase_early", "95% lower-bound"]))
ci_u_lm = float(np.exp(cph_lm.confidence_intervals_.loc["phase_early", "95% upper-bound"]))

lr_pan_lm = logrank_test(
    lm_df.loc[lm_df["phase_early"] == 1, "landmark_time"],
    lm_df.loc[lm_df["phase_early"] == 0, "landmark_time"],
    event_observed_A=lm_df.loc[lm_df["phase_early"] == 1, "landmark_event"],
    event_observed_B=lm_df.loc[lm_df["phase_early"] == 0, "landmark_event"],
)
p_lm = float(lr_pan_lm.p_value)

print(f"Pan-cancer landmark (stratified by cancer type):")
print(f"  HR = {hr_lm:.2f} [{ci_l_lm:.2f}-{ci_u_lm:.2f}], P = {p_lm:.4e}")
print(f"  n = {len(lm_df)}, post-landmark events = {int(lm_df['landmark_event'].sum())}")


## Save results


In [ ]:
def save_pkl(obj, name):
    with open(INTER_DIR / f"{name}.pkl", "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)


cox_summary = {
    "per_cancer": cox_results,
    "pan_cancer_HR": hr_pan,
    "pan_cancer_CI": (ci_lp, ci_up),
    "pan_cancer_P": p_pan,
    "loo_sensitivity": loo_results,
    "landmark_per_cancer": lm_per_cancer,
    "landmark_pan_HR": hr_lm,
    "landmark_pan_CI": (ci_l_lm, ci_u_lm),
    "landmark_pan_P": p_lm,
    "landmark_n": len(lm_df),
    "landmark_events": int(lm_df["landmark_event"].sum()),
}
save_pkl(cox_summary, "cox_summary")
print("\nNext: 05_wsi_validation.ipynb")
